# Subunit A interacting residues workflow

This notebook was executed using the following enviornment:

1. WSL Installation: https://learn.microsoft.com/en-us/windows/wsl/install
2. VS Code Installation: https://code.visualstudio.com/docs/setup/windows#_install-vs-code-on-windows
3. WSL extension on VS Code: https://code.visualstudio.com/docs/remote/wsl

OpenBabel installation:

1. ```sudo apt update```
2. ```sudo apt install openbabel```

Python environment:

1. ```!python -m venv .enzengdpa```
2. ```!pip install requirements.txt```

We evaluated the binding interactions between amino acids and a ligand using output files from different algorithms: AutoDock Vina, DiffDock, and Boltz-2. These algorithms were executed on the Tamarind Bio Platform.

We installed ```DiffDock-Pocket``` on WSL by following developers' instructions: https://github.com/plainerman/DiffDock-Pocket/

DiffDock-Pocket was used to validate the amino acids interacting with the ligand for each algorithm's output. It was executed separately from this notebook once the relevant amino acids were identified.

In [1]:
import os
os.environ['PYTHONDWRITEBYTECODE'] = '0'

# 0. Import packages

In [2]:
# Openbabel terminal subprocess
import subprocess
# Protein & Ligand Pre-Process
import prolif as plf
from rdkit import Chem
from rdkit.Chem import AllChem
import MDAnalysis as mda
# DataFrame Manipulation
import pandas as pd
# Molecular Visualization
from prolif.plotting.network import LigNetwork
import py3Dmol

/home/mreyes/gogec/2025/enzengDPA/.enzengdpa/lib/python3.12/site-packages/prolif/datafiles.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home/mreyes/gogec/2025/enzengDPA/.enzengdpa/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mreyes/gogec/2025/enzengDPA/.enzengdpa/lib/python3.12/site-packages/MDAnalysis/topology/tables.py:52: DeprecationWarning: Deprecated in version 2.8.0
MDAnalysis.topology.tables has been moved to MDAnalysis.guesser.tables. This import point will be removed in MDAnalysis version 3.0.0
  warnings.warn(wmsg, category=DeprecationWarn

# 1. Import AlphaFold-predicted structure

Add hydrogens to structure. 

Make sure to have installed ```openbabel```.

1. ```sudo apt update```
2. ```sudo apt install openbabel```

In [3]:
protein_raw_file = "tamarind_results/subunit_a_alphafold_model.pdb"
protein_hs_file = "interaction_results/subunit_a_with_h.pdb"

try:
    subprocess.run([
        "obabel", 
        protein_raw_file, 
        "-ipdb", 
        "-O", protein_hs_file, 
        "-opdb", 
        "-p", "7.4"
    ], check=True)
    print("Hydrogens added successfully!")
except subprocess.CalledProcessError as e:
    print(f"An error occurred while running OpenBabel: {e}")
except FileNotFoundError:
    print("OpenBabel is not installed or not in your PATH.")

Hydrogens added successfully!


*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is tamarind_results/subunit_a_alphafold_model.pdb)

1 molecule converted


Load structure with hydrogens into ProLIF

In [4]:
# Load the newly created protonated file
u = mda.Universe(protein_hs_file)

# Convert to ProLIF
dpa_synthase = plf.Molecule.from_mda(u)

print(f"Protein loaded with {dpa_synthase.n_residues} residues and explicit hydrogens.")

Protein loaded with 297 residues and explicit hydrogens.


# 2. Autodock Vina Results Analysis

Import ligand binding pose.

In [5]:
ligand_pose_file = "tamarind_results/subunit_a_ligand_autodock_vina.sdf"

lig_rdkit = Chem.SDMolSupplier(ligand_pose_file, removeHs=False)[0]
htpa_vina = plf.Molecule(lig_rdkit)
print(f"Ligand binding pose loaded.")

Ligand binding pose loaded.


Run fingerprint analysis.

In [6]:
fp = plf.Fingerprint()
fp.run_from_iterable([htpa_vina], dpa_synthase)

# Show results
df = fp.to_dataframe()
df.head()

100%|██████████| 1/1 [00:00<00:00, 47.31it/s]


ligand            UNK0                                                      \
protein        LEU32.A    VAL33.A    GLY34.A                       PHE35.A   
interaction VdWContact VdWContact HBAcceptor HBDonor VdWContact HBAcceptor   
Frame                                                                        
0                 True       True       True    True       True       True   

ligand                                                                       \
protein                    ASP36.A                       LYS47.A    ASN49.A   
interaction VdWContact Hydrophobic HBDonor VdWContact VdWContact VdWContact   
Frame                                                                         
0                 True        True    True       True       True       True   

ligand                             
protein        SER79.A    GLU81.A  
interaction VdWContact VdWContact  
Frame                              
0                 True       True

Display 2D diagram.

In [7]:
# Generate the network
net = LigNetwork.from_fingerprint(fp, htpa_vina, kind="frame", frame=0)

# Save the network to a file
html_file = "interaction_results/subunit_a_autodock_vina_interactions.html"
net.save(html_file)

print(f"File saved as {html_file}")

# Display
net.display()

File saved as interaction_results/subunit_a_autodock_vina_interactions.html


Retrieve Interacting Residues as a DataFrame.

In [8]:
interaction_counts_vina = df.T.sum(axis=1).reset_index()
interaction_counts_vina.columns = ['Ligand', 'Residue', 'Interaction', 'Count']

# Define the interactions
target_interactions = "HB|Hydrophobic|VdW"

# Filter the dataframe
filtered_interactions = interaction_counts_vina[
    interaction_counts_vina['Interaction'].str.contains(target_interactions, case=False)
]

# Sort by Residue or Count for better readability
filtered_interactions = filtered_interactions.sort_values(by=['Residue', 'Count'], ascending=[True, False])
filtered_interactions.to_csv("interaction_results/subunit_a_autodock_vina_interactions.csv",index=False)

# Display the result
print("Key Interactions (HBonds, Hydrophobic, VdW):")
display(filtered_interactions)

Key Interactions (HBonds, Hydrophobic, VdW):


,Ligand,Residue,Interaction,Count
11,UNK0,ASN49.A,VdWContact,1
7,UNK0,ASP36.A,Hydrophobic,1
8,UNK0,ASP36.A,HBDonor,1
9,UNK0,ASP36.A,VdWContact,1
13,UNK0,GLU81.A,VdWContact,1
2,UNK0,GLY34.A,HBAcceptor,1
3,UNK0,GLY34.A,HBDonor,1
4,UNK0,GLY34.A,VdWContact,1
0,UNK0,LEU32.A,VdWContact,1
10,UNK0,LYS47.A,VdWContact,1


Visualize 3D structure.

In [9]:
# Identify residues that had any interaction
interacting_res_list = interaction_counts_vina['Residue'].unique()
res_ids = [int(''.join(filter(str.isdigit, res))) for res in interacting_res_list]

# Open Protein Structure
with open(protein_hs_file, "r") as f:
    pdb_data = f.read()

# Open Ligand Structure
with open(ligand_pose_file, "r") as f:
    sdf_data = f.read()

# Initialize viewer
viewer = py3Dmol.view(width=800, height=600)
viewer.addModel(pdb_data, "pdb")
viewer.addModel(sdf_data, "sdf")

# 1. Style the Protein (Base color: Green)
viewer.setStyle({'model': 0}, {'cartoon': {'color': 'green', 'opacity': 0.8}})

# 2. Style the DpaA_N domain (Residues 6-123) - Color: Orange/Yellow
# We use a list range for the 'resi' parameter
viewer.addStyle({'model': 0, 'resi': [str(r) for r in range(6, 124)]}, 
                {'cartoon': {'color': '#FFA500'}}) # Orange hex

# 3. Style Interacting Residues (Red)
viewer.addStyle({'model': 0, 'resi': res_ids}, 
                {'stick': {'color': 'red'}, 'cartoon': {'color': 'red'}})

# 4. Style the Ligand (Blue)
viewer.setStyle({'model': 1}, {'stick': {'colorscheme': 'blueCarbon'}})

# 5. Add a Label for the DpaA_N domain
# We place the label at a central residue in the domain (e.g., residue 60)
viewer.addLabel("DpaA_N Domain", 
                {'fontSize': 14, 'fontColor': 'black', 'backgroundColor': 'orange', 'backgroundOpacity': 0.8},
                {'model': 0, 'resi': 60})

# Add labels for the interacting residues
for res_name in interacting_res_list:
    res_num = int(''.join(filter(str.isdigit, res_name)))
    viewer.addLabel(res_name, 
                    {'fontSize': 10, 'fontColor': 'white', 'backgroundColor': 'black'},
                    {'model': 0, 'resi': res_num})

# 6. Zoom to Ligand (Model 1) for publication-ready focus
viewer.zoomTo({'model': 1})
viewer.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# 3. DiffDock Results Analysis

Import ligand binding pose.

In [10]:
ligand_pose_file = "tamarind_results/subunit_a_ligand_diffdock.sdf"

lig_rdkit = Chem.SDMolSupplier(ligand_pose_file, removeHs=False)[0]
htpa_diffdock = plf.Molecule(lig_rdkit)
print(f"Ligand binding pose loaded.")

Ligand binding pose loaded.


Run fingerprint analysis.

In [11]:
fp = plf.Fingerprint()
fp.run_from_iterable([htpa_diffdock], dpa_synthase)

# Show results
df = fp.to_dataframe()
df.head()

100%|██████████| 1/1 [00:00<00:00, 75.37it/s]


ligand            UNK0                                 
protein       LYS118.A              ARG125.A   GLU290.A
interaction HBAcceptor VdWContact VdWContact VdWContact
Frame                                                  
0                 True       True       True       True

Display 2D diagram.

In [12]:
# Generate the network
net = LigNetwork.from_fingerprint(fp, htpa_vina, kind="frame", frame=0)

# Save the network to a file
html_file = "interaction_results/subunit_a_diffdock_interactions.html"
net.save(html_file)

print(f"File saved as {html_file}")

# Display
net.display()

File saved as interaction_results/subunit_a_diffdock_interactions.html


Retrieve Interacting Residues as a DataFrame.

In [13]:
interaction_counts_diffdock = df.T.sum(axis=1).reset_index()
interaction_counts_diffdock.columns = ['Ligand', 'Residue', 'Interaction', 'Count']

# Define the interactions
target_interactions = "HB|Hydrophobic|VdW"

# Filter the dataframe
filtered_interactions = interaction_counts_diffdock[
    interaction_counts_diffdock['Interaction'].str.contains(target_interactions, case=False)
]

# Sort by Residue or Count for better readability
filtered_interactions = filtered_interactions.sort_values(by=['Residue', 'Count'], ascending=[True, False])
filtered_interactions.to_csv("interaction_results/subunit_a_diffdock_interactions.csv",index=False)

# Display the result
print("Key Interactions (HBonds, Hydrophobic, VdW):")
display(filtered_interactions)

Key Interactions (HBonds, Hydrophobic, VdW):


,Ligand,Residue,Interaction,Count
2,UNK0,ARG125.A,VdWContact,1
3,UNK0,GLU290.A,VdWContact,1
0,UNK0,LYS118.A,HBAcceptor,1
1,UNK0,LYS118.A,VdWContact,1


Visualize 3D structure.

In [14]:
# Identify residues that had any interaction
interacting_res_list = interaction_counts_diffdock['Residue'].unique()
res_ids = [int(''.join(filter(str.isdigit, res))) for res in interacting_res_list]

# Open Protein Structure
with open(protein_hs_file, "r") as f:
    pdb_data = f.read()

# Open Ligand Structure
with open(ligand_pose_file, "r") as f:
    sdf_data = f.read()

# Initialize the viewer PDB file
viewer = py3Dmol.view(width=800, height=600)
viewer.addModel(pdb_data, "pdb")
viewer.addModel(sdf_data, "sdf")

# Style the Protein (Green); Cartoon Model
viewer.setStyle({'model': 0}, {'cartoon': {'color': 'green', 'opacity': 0.8}})

# Style the Ligand (Blue); Stick Model
viewer.setStyle({'model': 1}, {'stick': {'colorscheme': 'blueCarbon'}})

# Style Interacting Residues (Red)
viewer.addStyle({'model': 0, 'resi': res_ids}, 
                {'stick': {'color': 'red'}, 'cartoon': {'color': 'red'}})

# Add labels for the interacting residues
for res_name in interacting_res_list:
    res_num = int(''.join(filter(str.isdigit, res_name)))
    viewer.addLabel(res_name, 
                    {'fontSize': 10, 'fontColor': 'white', 'backgroundColor': 'black'},
                    {'model': 0, 'resi': res_num})

viewer.zoomTo()
viewer.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# 4. Boltz-2 Results Analysis

In [15]:
docked_pdb = "tamarind_results/subunit_a_complex_boltz.pdb" 

# Load the complex
u = mda.Universe(docked_pdb)

# Separate protein and ligand
ligand_atoms = u.select_atoms("not protein and not resname SOL WAT")

temp_lig_pdb = "interaction_results/aubunit_a_ligand_boltz.pdb"
ligand_atoms.atoms.write(temp_lig_pdb)

lig_rdkit_temp = Chem.MolFromPDBFile(temp_lig_pdb, removeHs=False)

ligand_pose_file = "interaction_results/subunit_a_ligand_boltz.sdf"
with Chem.SDWriter(ligand_pose_file) as writer:
    writer.write(lig_rdkit_temp)
print(f"Ligand successfully converted and stored as SDF.")

Ligand successfully converted and stored as SDF.


/home/mreyes/gogec/2025/enzengDPA/.enzengdpa/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:885: UserWarning: Unit cell dimensions not found. CRYST1 record set to unitary values.
  warnings.warn(
/home/mreyes/gogec/2025/enzengDPA/.enzengdpa/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(


Import ligand binding pose.

In [16]:
lig_rdkit = Chem.SDMolSupplier(ligand_pose_file, removeHs=False)[0]
htpa_boltz = plf.Molecule(lig_rdkit)
print(f"Ligand binding pose loaded.")

Ligand binding pose loaded.


Run fingerprint analysis.

In [17]:
fp = plf.Fingerprint()
fp.run_from_iterable([htpa_boltz], dpa_synthase)

# Show results
df = fp.to_dataframe()
df.head()

100%|██████████| 1/1 [00:00<00:00, 54.74it/s]


ligand            UNK0                                                         \
protein       PHE123.A   ASN132.A   THR136.A   GLY266.A   PRO268.A   GLY269.A   
interaction VdWContact VdWContact VdWContact VdWContact VdWContact HBAcceptor   
Frame                                                                           
0                 True       True       True       True       True       True   

ligand                  
protein                 
interaction VdWContact  
Frame                   
0                 True

Display 2D diagram.

In [18]:
# Generate the network
net = LigNetwork.from_fingerprint(fp, htpa_vina, kind="frame", frame=0)

# Save the network to a file
html_file = "interaction_results/subunit_a_boltz_interactions.html"
net.save(html_file)

print(f"File saved as {html_file}")

# Display
net.display()

File saved as interaction_results/subunit_a_boltz_interactions.html


Retrieve Interacting Residues as DataFrame.

In [19]:
interaction_counts_diffdock = df.T.sum(axis=1).reset_index()
interaction_counts_diffdock.columns = ['Ligand', 'Residue', 'Interaction', 'Count']

# Define the interactions
target_interactions = "HB|Hydrophobic|VdW"

# Filter the dataframe
filtered_interactions = interaction_counts_diffdock[
    interaction_counts_diffdock['Interaction'].str.contains(target_interactions, case=False)
]

# Sort by Residue or Count for better readability
filtered_interactions = filtered_interactions.sort_values(by=['Residue', 'Count'], ascending=[True, False])
filtered_interactions.to_csv("interaction_results/subunit_a_boltz_interactions.csv",index=False)

# Display the result
print("Key Interactions (HBonds, Hydrophobic, VdW):")
display(filtered_interactions)

Key Interactions (HBonds, Hydrophobic, VdW):


,Ligand,Residue,Interaction,Count
1,UNK0,ASN132.A,VdWContact,1
3,UNK0,GLY266.A,VdWContact,1
5,UNK0,GLY269.A,HBAcceptor,1
6,UNK0,GLY269.A,VdWContact,1
0,UNK0,PHE123.A,VdWContact,1
4,UNK0,PRO268.A,VdWContact,1
2,UNK0,THR136.A,VdWContact,1


Visualize 3D structure.

In [20]:
# Identify residues that had any interaction
interacting_res_list = interaction_counts_diffdock['Residue'].unique()
res_ids = [int(''.join(filter(str.isdigit, res))) for res in interacting_res_list]

# Open Protein Structure
with open(protein_hs_file, "r") as f:
    pdb_data = f.read()

# Open Ligand Structure
with open(ligand_pose_file, "r") as f:
    sdf_data = f.read()

# Initialize the viewer PDB file
viewer = py3Dmol.view(width=800, height=600)
viewer.addModel(pdb_data, "pdb")
viewer.addModel(sdf_data, "sdf")

# Style the Protein (Green); Cartoon Model
viewer.setStyle({'model': 0}, {'cartoon': {'color': 'green', 'opacity': 0.8}})

# Style the Ligand (Blue); Stick Model
viewer.setStyle({'model': 1}, {'stick': {'colorscheme': 'blueCarbon'}})

# Style Interacting Residues (Red)
viewer.addStyle({'model': 0, 'resi': res_ids}, 
                {'stick': {'color': 'red'}, 'cartoon': {'color': 'red'}})

# Add labels for the interacting residues
for res_name in interacting_res_list:
    res_num = int(''.join(filter(str.isdigit, res_name)))
    viewer.addLabel(res_name, 
                    {'fontSize': 10, 'fontColor': 'white', 'backgroundColor': 'black'},
                    {'model': 0, 'resi': res_num})

viewer.zoomTo()
viewer.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

END